### This demo showcases the implementation of story RSPY-808 (Implement STAC view of EDRS sessions)
See https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-808


In [37]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
import pprint
pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

auxip_client, cadip_client, catalog_client, staging_client, prip_client, edrs_client = init_demo()

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
EDRS service: http://rs-server-edrs:8000/edrs
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000


In [2]:
# STAC API landing page. /edrs/
edrs_client.get_landing()

{'type': 'Catalog',
 'id': 'stac-fastapi',
 'stac_version': '1.1.0',
 'description': 'Edrs collections of Copernicus Reference System Python',
 'links': [{'rel': 'self',
   'href': 'http://rs-server-edrs:8000/edrs/',
   'type': 'application/json'},
  {'rel': 'root',
   'href': 'http://rs-server-edrs:8000/edrs/',
   'type': 'application/json',
   'title': 'RS-PYTHON Edrs collections'},
  {'rel': 'data',
   'href': 'http://rs-server-edrs:8000/edrs/collections',
   'type': 'application/json',
   'title': 'Collections available for this Catalog'},
  {'rel': 'conformance',
   'href': 'http://rs-server-edrs:8000/edrs/conformance',
   'type': 'application/json',
   'title': 'STAC/OGC conformance classes implemented by this server'},
  {'rel': 'search',
   'href': 'http://rs-server-edrs:8000/edrs/search',
   'type': 'application/geo+json',
   'title': 'STAC search [GET]',
   'method': 'GET'},
  {'rel': 'search',
   'href': 'http://rs-server-edrs:8000/edrs/search',
   'type': 'application/geo+j

In [38]:
# STAC collections the user has permission to access. '/edrs/collections'
collections = edrs_client.get_collections()
for c in collections:
    pp.pprint(c.to_dict()['id'])

's1_pedc'
's2_pedc'


In [40]:
# Queryable fields. '/edrs/queryables'
general_queryables = edrs_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.pp(general_queryables)

{'$id': 'http://rs-server-edrs:8000/edrs/queryables',
 'type': 'object',
 'title': 'STAC Queryables.',
 '$schema': 'http://json-schema.org/draft-07/schema#',
 'properties': {'id': {'type': 'string',
                       'title': 'id',
                       'format': 'string',
                       'pattern': None,
                       'description': 'STAC Item id (session identifier)',
                       'enum': None},
                'collection': {'type': 'string',
                               'title': 'collection',
                               'format': 'string',
                               'pattern': None,
                               'description': 'Collection id',
                               'enum': None},
                'datetime': {'type': 'string',
                             'title': 'datetime',
                             'format': 'date-time',
                             'pattern': None,
                             'description': 'Nominal datetime

In [41]:
fields = list(general_queryables["properties"].keys())
print(fields)

['id', 'collection', 'datetime', 'start_datetime', 'end_datetime', 'published', 'platform', 'constellation']


In [46]:
collection_queryables = edrs_client.get_collection_queryables(collection_id='s1_pedc')
assert isinstance(collection_queryables, dict)
fields = list(collection_queryables["properties"].keys())
print(fields)

['id', 'collection', 'datetime', 'start_datetime', 'end_datetime', 'published', 'platform', 'constellation']


In [72]:
edrs_client.get_collection(collection_id='s1_pedc')

<CollectionClient id=s1_pedc>

In [47]:
item = edrs_client.get_item(collection_id='s1_pedc', item_id="DCS_02_202502131123000000987654")
item

<Item id=DCS_02_202502131123000000987654>

In [68]:
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=1, page=1)
items_list = list(items_iter)
pp.pprint(items_list)
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=1, page=2)
items_list = list(items_iter)
pp.pprint(items_list)

13:02:19.019 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'sortby': '+published', 'limit': 1, 'page': 1}.
13:02:19.171 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'sortby': '+published', 'limit': 1, 'page': 2}.


[<Item id=DCS_01_202501270945000000112233>]
[<Item id=DCS_02_202502131123000000987654>]


In [75]:
items = edrs_client.get_items(collection_id="s1_pedc", platform='sentinel-1c')
items_list = list(items)

13:06:49.473 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'platform': 'sentinel-1c'}.


In [77]:
import json
print(json.dumps(items_list[0].to_dict(), indent=2))

{
  "type": "Feature",
  "stac_version": "1.1.0",
  "stac_extensions": [
    "https://stac-extensions.github.io/file/v2.1.0/schema.json",
    "https://stac-extensions.github.io/timestamps/v1.1.0/schema.json"
  ],
  "id": "DCS_02_202502131123000000987654",
  "geometry": null,
  "properties": {
    "datetime": "2025-02-13T11:28:42Z",
    "start_datetime": "2025-02-13T11:23:00.000Z",
    "end_datetime": "2025-02-13T11:33:00.000Z",
    "platform": "sentinel-1c",
    "constellation": "sentinel-1",
    "published": "2025-02-13T11:28:42Z"
  },
  "links": [
    {
      "rel": "collection",
      "href": "http://rs-server-edrs:8000/edrs/collections/s1_pedc",
      "type": "application/json"
    },
    {
      "rel": "parent",
      "href": "http://rs-server-edrs:8000/edrs/collections/s1_pedc",
      "type": "application/json"
    },
    {
      "rel": "root",
      "href": "http://rs-server-edrs:8000/edrs/",
      "type": "application/json",
      "title": "RS-PYTHON Edrs collections"
    },
  

In [79]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2026-01-01T00:00:00Z/..")
items_list = list(items)
assert not items_list

13:07:40.399 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2026-01-01T00:00:00Z/..'}.


In [80]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-01-01T00:00:00Z/..")
items_list = list(items)
assert len(items_list) == 2
items_list

13:07:49.045 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2024-01-01T00:00:00Z/..'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [24]:
items_list[0]

<Item id=DCS_02_202502131123000000987654>

In [81]:
items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="start_datetime='2025-02-13T11:23:00Z'",
)
items_list = list(items)

13:08:14.332 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "start_datetime='2025-02-13T11:23:00Z'"}.


In [82]:
items_list

[<Item id=DCS_02_202502131123000000987654>]

In [83]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-02-13T11:23:00Z/2025-02-13T11:33:00Z")
items_list = list(items)
assert len(items_list) == 2
items_list

13:08:25.885 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2024-02-13T11:23:00Z/2025-02-13T11:33:00Z'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [84]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="../2025-02-13T12:00:00Z")
items_list = list(items)
assert len(items_list) == 2
items_list

13:08:28.673 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '../2025-02-13T12:00:00Z'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [85]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-01-01T00:00:00Z/..")
items_list = list(items)
assert len(items_list) == 2
items_list

13:08:33.022 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2024-01-01T00:00:00Z/..'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [86]:
items = edrs_client.get_items(
    collection_id="s1_pedc",
    start_datetime='2025-02-13T11:23:00.000Z',
)
items_list = list(items)

13:08:35.504 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'start_datetime': '2025-02-13T11:23:00.000Z'}.


In [87]:
items = edrs_client.get_items(collection_id='s1_pedc', filter="platform='sentinel-1c'")
items_list = list(items)
len(items_list)

13:08:46.595 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "platform='sentinel-1c'"}.


1

In [88]:
items = edrs_client.get_items(collection_id='s1_pedc', filter="constellation='sentinel-1'")
items_list = list(items)
len(items_list)


13:08:47.892 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "constellation='sentinel-1'"}.


2

In [89]:
items_iter = edrs_client.get_items(
    collection_id='s1_pedc',
    filter="platform='sentinel-1c' AND constellation='sentinel-1'",
    sortby='-published',
    limit=2,
    page=1,
)
items_list = list(items_iter)
len(items_list)

13:09:00.189 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "platform='sentinel-1c' AND constellation='sentinel-1'", 'sortby': '-published', 'limit': 2, 'page': 1}.


1

In [90]:
# cql2-json filter
import json
items_iter = edrs_client.get_items(
    collection_id="s1_pedc",
    **{
        "filter-lang": "cql2-json",
        "filter": json.dumps({
            "op": "=",
            "args": [
                {"property": "published"},
                {"literal": "2025-02-13T11:28:42Z"},
            ],
        }),
    },
)
items_list = list(items_iter)

13:09:25.588 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter-lang': 'cql2-json', 'filter': '{"op": "=", "args": [{"property": "published"}, {"literal": "2025-02-13T11:28:42Z"}]}'}.


In [35]:
import json

items_iter = edrs_client.get_items(
    collection_id="s1_pedc",
    **{
        "filter-lang": "cql2-json",
        "filter": json.dumps({
            "op": "and",
            "args": [
                {
                    "op": "=",
                    "args": [
                        {"property": "platform"},
                        {"literal": "sentinel-1c"},
                    ],
                },
                {
                    "op": "=",
                    "args": [
                        {"property": "constellation"},
                        {"literal": "sentinel-1"},
                    ],
                },
            ],
        }),
    },
)
items_list = list(items_iter)


10:43:45.488 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter-lang': 'cql2-json', 'filter': '{"op": "and", "args": [{"op": "=", "args": [{"property": "platform"}, {"literal": "sentinel-1c"}]}, {"op": "=", "args": [{"property": "constellation"}, {"literal": "sentinel-1"}]}]}'}.


In [91]:
items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="published='2025-02-13T11:28:42Z'",
)
list(items)

items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="start_datetime='2025-02-13T11:23:00.000Z'",
)
list(items)

items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="end_datetime='2025-02-13T11:33:00.000Z'",
)
list(items)

items = edrs_client.get_items(
    collection_id="s1_pedc",
    datetime="2025-02-13T11:23:00Z/2025-02-13T11:33:00Z",
)
list(items)


13:09:52.347 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "published='2025-02-13T11:28:42Z'"}.
13:09:52.488 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "start_datetime='2025-02-13T11:23:00.000Z'"}.
13:09:52.613 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "end_datetime='2025-02-13T11:33:00.000Z'"}.
13:09:52.733 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2025-02-13T11:23:00Z/2025-02-13T11:33:00Z'}.


[<Item id=DCS_02_202502131123000000987654>]